# 06 · nn.Module and the training loop

Pairs with `GUIDE.md` steps 5-8. Build a model, watch gradients flow, run a loop, and
sanity-check by overfitting a tiny subset. Runs on CPU (small + synthetic).

In [ ]:
import torch
from gpulab.learn import inspect as I
HAS_CUDA = torch.cuda.is_available()
dev = torch.device("cuda" if HAS_CUDA else "cpu")
print("device:", dev, "| CUDA:", HAS_CUDA)
if not HAS_CUDA:
    print("No CUDA here - cells run on CPU; the timing/memory numbers are only")
    print("meaningful on your RTX 3060. Run this notebook there for the real story.")

In [ ]:
import numpy as np
from torch import nn
# Synthetic 3-class dataset of length-120 curves (stands in for the real melt data).
rng = np.random.default_rng(0)
def make(n=600):
    y = rng.integers(0, 3, n)
    t = np.linspace(0, 1, 120)[None, :]
    centers = np.array([0.3, 0.5, 0.7])[y][:, None]
    X = np.exp(-((t - centers) ** 2) / 0.01) + rng.normal(0, 0.05, (n, 120))
    return X.astype("float32"), y.astype("int64")
Xtr, ytr = make(600); Xte, yte = make(200)
print(Xtr.shape, ytr.shape)

## 1. Build a tiny module yourself, then inspect it

In [ ]:
# Your turn: write a minimal classifier (Linear(120, 3) is enough to start).
# class MLP(nn.Module):
#     def __init__(self): ...
#     def forward(self, x): ...
# model = MLP().to(dev)
# For now, use the repo's CNN as a working stand-in:
from gpulab.models.cnn1d import CNN1D
model = CNN1D(n_classes=3, roi_len=120, head="flatten").to(dev)
I.param_table(model)
print("state_dict keys:", list(model.state_dict().keys())[:4], "...")

## 2. One forward/backward — does gradient reach every layer?

In [ ]:
import matplotlib; matplotlib.use("Agg")
from gpulab.learn import viz
xb = torch.tensor(Xtr[:64], device=dev); yb = torch.tensor(ytr[:64], device=dev)
loss = nn.CrossEntropyLoss()(model(xb), yb)
loss.backward()
I.grad_norms(model)
viz.plot_grad_flow(model)   # returns a Figure
print("loss", loss.item())

## 3. The loop — write it, then diff against the repo

In [ ]:
# Your turn: implement train_one_epoch(model, X, y): shuffle -> minibatch ->
# zero_grad -> forward -> loss -> backward -> step. Then compare to gpulab.train.loop.
# Working reference so this notebook runs end-to-end:
from gpulab.train.loop import TrainConfig, train
out = train(model, Xtr, ytr, Xte, yte, TrainConfig(epochs=8, batch_size=128, amp=False))
viz.plot_history(out["history"])
print("final test acc:", round(out["final_test_acc"], 3))

## 4. Overfit-a-tiny-subset sanity check

In [ ]:
# The single most useful DL debug: a good model+loop can memorize a few samples.
tiny = CNN1D(n_classes=3, roi_len=120, head="flatten").to(dev)
o = train(tiny, Xtr[:50], ytr[:50], Xtr[:50], ytr[:50], TrainConfig(epochs=60, batch_size=50, amp=False))
print("train acc on 50 samples (want ~1.0):", round(o["final_test_acc"], 3))

> **Concepts to note** (copy into your own theory notebook):
> - Parameters register automatically when assigned as `nn.Module` attributes.
> - Every trainable param must get a non-None, non-zero grad after backward.
> - The loop is always: zero_grad -> forward -> loss -> backward -> step.
> - If it can't overfit 50 samples, the model/loop is broken - fix that first.
> - Save `state_dict` (weights), not the pickled model object.